# Resample LIDAR to MintPy

This notebook resamples ASO GeoTIFFs onto a MintPy geocoded grid, writes the resampled rasters into `outputs/resampled/`, and then packages them into a MintPy-style HDF5 timeseries.

In [ ]:
from pathlib import Path

from snowsar.utils.lidar_utils import (
    build_lidar_timeseries_h5,
    read_geotiff_stack_sorted_by_date,
    resample_many_geotiffs,
)


In [ ]:
NOTEBOOK_DIR = Path.cwd() / "LIDAR_notebooks" if (Path.cwd() / "LIDAR_notebooks").exists() else Path.cwd()
OUTPUT_DIR = NOTEBOOK_DIR / "outputs_2022"
RESAMPLED_DIR = OUTPUT_DIR / "resampled"
RESAMPLED_DIR.mkdir(parents=True, exist_ok=True)

TIMESERIES_FILE = Path("/Volumes/Fortress_L3/SnowWaterEquivalent/ALOS2_CA_P67_F750_A/CA_P67_F750_A_Dec21_June22/mintpy/geo/geo_timeseries_ERA5_demErr.h5")
ASO_INPUT_DIR = Path("/Volumes/Fortress_L3/SnowWaterEquivalent/ALOS2_CA_P67_F750_A/ASO")
YEAR = "2022"
ASO_GLOB = "*snowdepth_3m*.tif"
OUTPUT_TIMESERIES_FILE = OUTPUT_DIR / "timeseries_lidar_swe.h5"
OVERWRITE_RESAMPLED = False

if not TIMESERIES_FILE.exists():
    raise FileNotFoundError(f"Update the config cell. Missing MintPy file: {TIMESERIES_FILE}")
if not ASO_INPUT_DIR.exists():
    raise FileNotFoundError(f"Update the config cell. Missing ASO directory: {ASO_INPUT_DIR}")

aso_files = sorted((ASO_INPUT_DIR / YEAR).rglob(ASO_GLOB))
if not aso_files:
    raise FileNotFoundError(f"No ASO rasters matched {(ASO_INPUT_DIR / YEAR) / ASO_GLOB}")

aso_files[:]


## Resample source rasters

In [ ]:
written_paths = resample_many_geotiffs(
    aso_files,
    TIMESERIES_FILE,
    output_dir=RESAMPLED_DIR,
    overwrite=OVERWRITE_RESAMPLED,
)
written_paths[:5]


## Inspect stack metadata

In [ ]:
date_list, stack, metadata = read_geotiff_stack_sorted_by_date(str(RESAMPLED_DIR / "resampled_*.tif"))
{
    "num_dates": len(date_list),
    "first_date": date_list[0],
    "last_date": date_list[-1],
    "stack_shape": stack.shape,
    "first_path": metadata["paths"][0],
}


## Build MintPy-style LIDAR timeseries

In [ ]:
output_path = build_lidar_timeseries_h5(
    str(RESAMPLED_DIR / "resampled_*.tif"),
    TIMESERIES_FILE,
    OUTPUT_TIMESERIES_FILE,
)
output_path
